# G2 — ViT 확장 bridge (DeiT-Tiny / CIFAR100 / W3)

G1(CNN)과 **병렬 Colab 세션**으로. ViT는 인프라가 달라 self-contained:
- FP = DeiT-Tiny **pretrained + AdamW finetune** (224 upsample) — `train_baseline`(ResNet SGD) 못 씀
- 프로토콜은 *light* (224라 단일층 full sweep 느림): **PTQ gap + normHd2 proxy + subset payoff**
- quant 엔진은 Linear 양자화 됨(`ptq`가 Conv+Linear swap) → proxy/payoff는 generic 엔진 그대로

= "ViT서도 partial QAT payoff 되나" bridge. **현상 재현이지 recipe 제안 아님.** (FP 품질·224 속도가 리스크.)

In [ ]:
# --- Colab 셋업 ---
import os
REPO = '/content/26_Capstone'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/u-nsiq/26_Capstone.git {REPO}
else:
    !git -C {REPO} pull -q
os.chdir(REPO)
!pip install -q -r requirements.txt
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# --- 엔진 + Drive + 경로 ---
from qat_engine import *
import numpy as np, json, timm
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

try:
    from google.colab import drive; drive.mount('/content/drive')
    ART = '/content/drive/MyDrive/26_Capstone'
except Exception:
    ART = './_local_art'
os.makedirs(f'{ART}/checkpoints', exist_ok=True); os.makedirs(f'{ART}/outputs/G1', exist_ok=True)
N_BITS = 3
print('device', DEVICE)

In [ ]:
# --- ViT FP: DeiT-Tiny pretrained + AdamW finetune (224 upsample CIFAR100, 캐시) ---
IMG = 224
imnorm = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))   # ImageNet norm (pretrained 일관)
tf_tr = transforms.Compose([transforms.Resize(IMG), transforms.RandomCrop(IMG, padding=16),
                            transforms.RandomHorizontalFlip(), transforms.ToTensor(), imnorm])
tf_te = transforms.Compose([transforms.Resize(IMG), transforms.ToTensor(), imnorm])
DR = f'{ART}/data'
tr_set = datasets.CIFAR100(DR, train=True,  download=True, transform=tf_tr)
te_set = datasets.CIFAR100(DR, train=False, download=True, transform=tf_te)
ca_set = Subset(datasets.CIFAR100(DR, train=True, download=True, transform=tf_te), list(range(512)))
train224 = DataLoader(tr_set, batch_size=128, shuffle=True,  num_workers=2, drop_last=True)
val224   = DataLoader(te_set, batch_size=128, shuffle=False, num_workers=2)
calib224 = DataLoader(ca_set, batch_size=64,  shuffle=False, num_workers=2)

VCKPT = f'{ART}/checkpoints/deit_tiny_cifar100_fp.pt'
vit = timm.create_model('deit_tiny_patch16_224', pretrained=True, num_classes=100).to(DEVICE)
if os.path.exists(VCKPT):
    vit.load_state_dict(torch.load(VCKPT, map_location='cpu')); vit = vit.to(DEVICE)
    print('ViT FP 캐시 로드')
else:
    EPOCHS = 5                       # 느리면 3으로. pretrained라 빨리 올라옴
    print(f'ViT finetune (AdamW, {EPOCHS}ep, 224)...')
    opt = torch.optim.AdamW(vit.parameters(), lr=1e-4, weight_decay=0.05)
    crit = nn.CrossEntropyLoss()
    for ep in range(EPOCHS):
        vit.train()
        for x, y in train224:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(); loss = crit(vit(x), y); loss.backward(); opt.step()
        a, _ = evaluate_full(vit, val224, DEVICE); print(f'  ep{ep} val acc {a:.2f}')
    torch.save(vit.state_dict(), VCKPT)
vit.eval()
fp_acc, _ = evaluate_full(vit, val224, DEVICE)
print(f'ViT FP acc {fp_acc:.2f}')

In [ ]:
# --- PTQ gap + proxy(normHd2) ---  (엔진 generic: Linear도 양자화/proxy 됨)
qm = make_ptq_model(vit, N_BITS, DEVICE)
ptq_acc, L_PTQ = evaluate_full(qm, val224, DEVICE)
layers = list_quant_layers(qm)
costs = get_layer_costs(qm, layers)
scores = proxy_scores(qm, vit, N_BITS, calib224, layers=layers)
print(f'ViT: {len(layers)} quant layers | FP {fp_acc:.2f} -> PTQ {ptq_acc:.2f} (gap {fp_acc-ptq_acc:.2f})')

In [ ]:
# --- subset payoff (normHd2-topk vs random vs full) — light (seed2, t<=300) ---
B = 0.25; rng = np.random.default_rng(0)
rules = {'PTQ-only': [], 'full': list(layers),
         'normHd2-topk': select_subset(scores, costs, B, by='normHd2'),
         'random':       select_subset({n: float(rng.random()) for n in layers}, costs, B)}
for r, ls in rules.items():
    pr = sum(get_layer_costs(vit, ls).values()) / sum(costs.values()) if ls else 0.0
    print(f'  {r:<14} k={len(ls):<3} param={pr*100:5.1f}%')
u5 = run_u5_subset(vit, N_BITS, rules, train224, val224, steps=300, eval_at=(30, 100, 300), seeds=(0, 1), device=DEVICE)
print('payoff done, L_PTQ', round(u5['L_PTQ'], 4))

In [ ]:
# --- 표 + 저장 ---
T = u5['t_evals'][-1]
def recov(a, p, f):
    return 100.0 * (a - p) / (f - p) if f > p else float('nan')
fl = u5['acc']['full'][T]
print(f'DeiT-Tiny / W{N_BITS}   (FP {fp_acc:.2f}, PTQ {ptq_acc:.2f})')
print(f'  rule           acc@{T}   param%   recov%')
for r in rules:
    a = u5['acc'][r][T]; pr = u5['param_ratio'][r] * 100
    print(f'  {r:<13} {a:6.2f}   {pr:5.1f}   {recov(a, ptq_acc, fl):5.1f}')
out = dict(model='deit_tiny', fp_acc=float(fp_acc), ptq_acc=float(ptq_acc), gap=float(fp_acc - ptq_acc),
           n_layers=len(layers), B=B, payoff=u5, subsets={r: rules[r] for r in rules})
json.dump(out, open(f'{ART}/outputs/G1/g2_vit_bridge_w{N_BITS}.json', 'w'), indent=2)
print('saved g2_vit_bridge')

## G2 해석 (ViT bridge)

- **`normHd2-topk`가 적은 param%로 full에 근접** → ViT서도 partial QAT payoff 재현 → 발표 "CNN→ViT까지 확장".
- **`normHd2-topk` > `random`?** → ViT 단계서도 단기proxy가 *약하게라도* 도움 되는지 (U5와 같은 질문).

⚠️ 한계(정직):
- **단일층 진단(normHd2→short 상관) 생략** — 224라 ~50 Linear층 sweep이 너무 느림. ViG2는 *적용 bridge*지 full 진단 아님.
- **FP = pretrained finetune** (CIFAR-100 from-scratch ViT는 어려움). FP acc가 낮으면 bridge 약함.
- recovery는 SGD momentum-0(연구 프로토콜 일관)이라 ViT엔 보수적일 수 있음.

→ 발표선 **"ViT 확장 *시작*: payoff 재현 신호"** 정도. full ViT 진단은 후속(ViT-튜닝 학습 + 경량 단일층).

*트림: 느리면 cell3 `EPOCHS=3`, cell5 `seeds=(0,)`·`steps=100`.*